# Winkansen van voetbal wedstrijden voorspellen
#### Een onderzoek van Lucas Hoetink, Hidde monsma, Khaleel Al-Aqel, Gamal Al-Sabaeei

In dit rapport gaan wij de winkansen van voetbalwedstrijden voorspellen. Dit gaan we doen op basis van principes uit de data science en doormiddel van machine learning modellen.

---

## Importing Libraries

Als eerste moeten we altijd de juiste libraries importeren zodat we toegang hebben tot de benodigde tools.

In [ ]:
import sqlite3 as sql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import subprocess
import os
from pathlib import Path
import tempfile

---

## Loading Data

Als volgende moeten we de data ophalen en inladen uit de `SQL` database

In [ ]:
subprocess.run(['pip', 'install', 'kaggle', '-q'], capture_output=True)

# Set up Kaggle credentials
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)

credentials = {
    "username": "lucashoetink",
    "key": "22e34cd52b72413f58087cacec1636c7"
}

with open(kaggle_dir / 'kaggle.json', 'w') as f:
    json.dump(credentials, f)

os.chmod(kaggle_dir / 'kaggle.json', 0o600)

# Download to a writable temp directory
temp_dir = tempfile.gettempdir()
os.system(f'kaggle datasets download -d hugomathien/soccer --unzip -p {temp_dir}')

# Connect to the database from temp directory
db_path = os.path.join(temp_dir, 'database.sqlite')
connection = sql.connect(db_path)
print(f"Connected to database at: {db_path}")

Dataset URL: https://www.kaggle.com/datasets/hugomathien/soccer
License(s): ODbL-1.0


100%|█████████████████████████████████████| 32.7M/32.7M [00:02<00:00, 14.6MB/s]



Connected to database at: /var/folders/p9/vd2t0bjd711gt4c08zmm0w4c0000gp/T/database.sqlite


---

## SQL Exploration

In [9]:
# See all table names
cursor = connection.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Available tables:", tables)

Available tables: [('sqlite_sequence',), ('Player_Attributes',), ('Player',), ('Match',), ('League',), ('Country',), ('Team',), ('Team_Attributes',)]


In [11]:
# Load any table into a DataFrame
df = pd.read_sql("SELECT * FROM Match LIMIT 5", connection)
display(df.head())

,id,country_id,league_id,season,stage,date,match_api_id,home_team_api_id,away_team_api_id,home_team_goal,...,SJA,VCH,VCD,VCA,GBH,GBD,GBA,BSH,BSD,BSA
0,1,1,1,2008/2009,1,2008-08-17 00:00:00,492473,9987,9993,1,...,4.00,1.65,3.40,4.50,1.78,3.25,4.00,1.73,3.40,4.20
1,2,1,1,2008/2009,1,2008-08-16 00:00:00,492474,10000,9994,0,...,3.80,2.00,3.25,3.25,1.85,3.25,3.75,1.91,3.25,3.60
2,3,1,1,2008/2009,1,2008-08-16 00:00:00,492475,9984,8635,0,...,2.50,2.35,3.25,2.65,2.50,3.20,2.50,2.30,3.20,2.75
3,4,1,1,2008/2009,1,2008-08-17 00:00:00,492476,9991,9998,5,...,7.50,1.45,3.75,6.50,1.50,3.75,5.50,1.44,3.75,6.50
4,5,1,1,2008/2009,1,2008-08-16 00:00:00,492477,7947,9985,1,...,1.73,4.50,3.40,1.65,4.50,3.50,1.65,4.75,3.30,1.67
